In [17]:
import os
import pandas as pd
import numpy as np
import warnings

# ==============================================================================
# 0. CONTROL D'AVISOS (WARNINGS)
# ==============================================================================
# Silenciem els avisos d'openpyxl sobre la validació de dades dels Excels
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# ==============================================================================
# 1. CONFIGURACIÓ DE CARPETES
# ==============================================================================
carpeta_entrada = '../data/raw/'
carpeta_sortida = '../data/processed/netejats/'
os.makedirs(carpeta_sortida, exist_ok=True)

noms_districte = ['Codi_districte', 'Codi_Districte', 'Districte', 'codi_districte']
noms_seccio = ['Seccio_censal', 'Seccio_Censal', 'Codi_SC', 'Seccio', 'Secció', 'codi_seccio']

def trobar_columna(df, llista_noms):
    for col in df.columns:
        if col.strip() in llista_noms: return col.strip()
    return None

# ==============================================================================
# 2. LES TEVES PRENETEGES ESPECÍFIQUES (BUSINESS LOGIC)
# ==============================================================================

def regla_avis_sols(df):
    """(Fila 7) Filtra i suma NOMÉS les llars amb 1 persona de més de 75 anys."""
    print("      -> Aplicant regla: Extracció d'avis que viuen sols (DOM_65_99 == 1)...")
    col_categoria = trobar_columna(df, ['DOM_65_99', 'Nombre_persones'])
    col_valor = trobar_columna(df, ['Valor', 'Nombre', 'Total'])
    
    if col_categoria and col_valor:
        df[col_categoria] = df[col_categoria].astype(str).str.strip()
        df_filtrat = df[df[col_categoria] == '1'].copy()
        df_aplanat = df_filtrat.groupby('CODI_UNIC')[col_valor].sum().reset_index()
        df_aplanat.rename(columns={col_valor: 'Llars_1_Avi_Sol'}, inplace=True)
        return df_aplanat
    return df

def regla_edificacions_ponderades(df):
    """(Fila 8) Calcula l'Any Mitjà Ponderat i el Total d'Edificacions per secció."""
    print("      -> Aplicant regla: Càlcul d'Any Ponderat i Total Edificis...")
    col_any = trobar_columna(df, ['Any_construccio', 'Any_Construccio', 'Rang_any'])
    col_num = trobar_columna(df, ['Nombre', 'Valor'])
    
    if col_any and col_num:
        def parse_any(x):
            x = str(x).strip()
            if x == 'No consta': return np.nan
            if x.startswith('<'): return int(x.replace('<', '')) - 1
            if ' i posterior' in x: return int(x.split(' ')[0]) + 2
            if '-' in x:
                parts = x.split('-')
                if parts[0].isdigit() and parts[1].isdigit():
                    return (int(parts[0]) + int(parts[1])) / 2
            if x.isdigit(): return float(x)
            return np.nan

        df['Any_Numeric'] = df[col_any].apply(parse_any)
        df_net = df.dropna(subset=['Any_Numeric', col_num]).copy()
        
        df_net['Ponderacio'] = df_net['Any_Numeric'] * df_net[col_num]
        
        df_aplanat = df_net.groupby('CODI_UNIC').agg(
            Total_Edificacions=(col_num, 'sum'),
            Suma_Anys_Pond=('Ponderacio', 'sum')
        ).reset_index()
        
        df_aplanat['Any_Mitja_Ponderat'] = round(df_aplanat['Suma_Anys_Pond'] / df_aplanat['Total_Edificacions'], 1)
        df_aplanat.drop(columns=['Suma_Anys_Pond'], inplace=True)
        
        return df_aplanat
    return df

# Connectem els fitxers amb la seva preneteja
REGLES_PRENETEJA = {
    '2025_pad_dom_mdbas_tipus-domicili_edat.csv': regla_tipus_domicili,
    '2025_pad_dom_mdbas_edat-6599.csv': regla_avis_sols,
    '2026_edificacions_any_con.csv': regla_edificacions_ponderades
}

# ==============================================================================
# 3. MOTOR PRINCIPAL
# ==============================================================================
fitxers = [f for f in os.listdir(carpeta_entrada) if f.endswith('.csv') or f.endswith('.xlsx')]
print(f"⚙️ Iniciant procés per a {len(fitxers)} fitxers...\n")

for arxiu in fitxers:
    if arxiu == 'pad_dimensions.csv': continue # Saltem l'arxiu diccionari
    
    ruta_completa = os.path.join(carpeta_entrada, arxiu)
    try:
        # LECTURA AMB LOW_MEMORY=FALSE PER EVITAR AVISOS DTYPE
        if arxiu.endswith('.csv'):
            try:
                df = pd.read_csv(ruta_completa, low_memory=False)
                if len(df.columns) == 1: 
                    df = pd.read_csv(ruta_completa, sep=';', low_memory=False)
            except:
                df = pd.read_csv(ruta_completa, sep=';', low_memory=False)
        else:
            df = pd.read_excel(ruta_completa)
            
        col_dist = trobar_columna(df, noms_districte)
        col_sec = trobar_columna(df, noms_seccio)
        
        if arxiu == '2025_consum_electricitat_BCN.csv':
             print(f"⚠️ SALTAT INTENCIONADAMENT: {arxiu} (Va per Codi Postal)")
             continue
             
        if col_dist and col_sec:
            df[col_dist] = pd.to_numeric(df[col_dist], errors='coerce').fillna(0).astype(int)
            df[col_sec] = pd.to_numeric(df[col_sec], errors='coerce').fillna(0).astype(int)
            
            dist_str = df[col_dist].astype(str).str.zfill(2)
            sec_str = df[col_sec].astype(str).str.zfill(3).str[-3:]
            
            df['CODI_UNIC'] = '08019' + dist_str + sec_str
            cols = ['CODI_UNIC'] + [c for c in df.columns if c != 'CODI_UNIC']
            df = df[cols]
            
            missatge_estat = f"✅ OK: {arxiu}"
            if arxiu in REGLES_PRENETEJA:
                print(f"🛠️ PRENETEJA DETECTADA per a: {arxiu}")
                funcio = REGLES_PRENETEJA[arxiu]
                df = funcio(df)
                missatge_estat = f"✨ REPARAT I APLANAT: {arxiu}"

            nom_sortida = arxiu.replace('.xlsx', '.csv') 
            ruta_sortida = os.path.join(carpeta_sortida, nom_sortida)
            df.to_csv(ruta_sortida, index=False)
            print(missatge_estat + "\n")
        else:
            print(f"❌ ERROR: {arxiu} -> Sense columnes geogràfiques.\n")
            
    except Exception as e:
        print(f"🚨 ERROR CRÍTIC amb {arxiu}: {e}\n")

print("🚀 Procés 02 Finalitzat sense avisos de memòria. Pots llançar el Mega-Merge (Script 03).")

⚙️ Iniciant procés per a 15 fitxers...

✅ OK: 2022_renda_disponible_llars_per_persona.csv

✅ OK: 2023_atles_renda_bruta_llar.csv

✅ OK: 2023_atles_renda_mitjana.csv

⚠️ SALTAT INTENCIONADAMENT: 2025_consum_electricitat_BCN.csv (Va per Codi Postal)
🛠️ PRENETEJA DETECTADA per a: 2025_pad_dom_mdbas_edat-6599.csv
      -> Aplicant regla: Extracció d'avis que viuen sols (DOM_65_99 == 1)...
✨ REPARAT I APLANAT: 2025_pad_dom_mdbas_edat-6599.csv

🛠️ PRENETEJA DETECTADA per a: 2026_edificacions_any_con.csv
      -> Aplicant regla: Càlcul d'Any Ponderat i Total Edificis...
✨ REPARAT I APLANAT: 2026_edificacions_any_con.csv

✅ OK: 2026_edificacions_edat_mitjana.csv

✅ OK: 2026_edificacions_superficie.csv

❌ ERROR: adreces_per_seccio_censal.csv -> Sense columnes geogràfiques.

❌ ERROR: DADES_BIBLIOGRAFIA.xlsx -> Sense columnes geogràfiques.

❌ ERROR: dades_prova.xlsx -> Sense columnes geogràfiques.

❌ ERROR: OD_Arbrat_Parcs_BCN.csv -> Sense columnes geogràfiques.

🚨 ERROR CRÍTIC amb opendatabcn_NP